Работа с табличными данными

In [574]:
import pandas as pd
import numpy as np
import copy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from dotenv import load_dotenv
import wandb
import os
import logging
import time
import torch.nn as nn
import random
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import *
import pickle

In [575]:
df = pd.read_csv('dataset/train.csv', sep="|")
df

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition,fraud
0,5,1054,54.70,7,0,3,0.027514,0.051898,0.241379,0
1,3,108,27.36,5,2,4,0.129630,0.253333,0.357143,0
2,3,1516,62.16,3,10,5,0.008575,0.041003,0.230769,0
3,6,1791,92.31,8,4,4,0.016192,0.051541,0.275862,0
4,5,430,81.53,3,7,2,0.062791,0.189605,0.111111,0
...,...,...,...,...,...,...,...,...,...,...
1874,1,321,76.03,8,7,2,0.071651,0.236854,0.347826,0
1875,1,397,41.89,5,5,0,0.065491,0.105516,0.192308,1
1876,4,316,41.83,5,8,1,0.094937,0.132373,0.166667,0
1877,2,685,62.68,1,6,2,0.035036,0.091504,0.041667,0


In [576]:
df.dtypes

trustLevel                     int64
totalScanTimeInSeconds         int64
grandTotal                   float64
lineItemVoids                  int64
scansWithoutRegistration       int64
quantityModifications          int64
scannedLineItemsPerSecond    float64
valuePerSecond               float64
lineItemVoidsPerPosition     float64
fraud                          int64
dtype: object

In [577]:
df.isna().sum()

trustLevel                   0
totalScanTimeInSeconds       0
grandTotal                   0
lineItemVoids                0
scansWithoutRegistration     0
quantityModifications        0
scannedLineItemsPerSecond    0
valuePerSecond               0
lineItemVoidsPerPosition     0
fraud                        0
dtype: int64

In [578]:
df.fraud.value_counts()

fraud
0    1775
1     104
Name: count, dtype: int64

In [579]:
y = df["fraud"]
X = df.drop(columns=["fraud"])

In [580]:
df.describe()

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition,fraud
count,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000
mean,3.401809,932.153273,50.864492,5.469931,4.904204,2.525279,0.058138,0.201746,0.745404,0.055349
std,1.709404,530.144640,28.940202,3.451169,3.139697,1.695472,0.278512,1.242135,1.327241,0.228720
min,1.000000,2.000000,0.010000,0.000000,0.000000,0.000000,0.000548,0.000007,0.000000,0.000000
25%,2.000000,474.500000,25.965000,2.000000,2.000000,1.000000,0.008384,0.027787,0.160000,0.000000
50%,3.000000,932.000000,51.210000,5.000000,5.000000,3.000000,0.016317,0.054498,0.350000,0.000000
75%,5.000000,1397.000000,77.285000,8.000000,8.000000,4.000000,0.032594,0.107313,0.666667,0.000000
max,6.000000,1831.000000,99.960000,11.000000,10.000000,5.000000,6.666667,37.870000,11.000000,1.000000


## Предобработка данных

In [581]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [582]:
X_train.shape

(1503, 9)

In [583]:
X_val.shape

(376, 9)

In [584]:
X_test = pd.read_csv("dataset/test.csv", sep="|")
X_test

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition
0,4,467,88.48,4,8,4,0.014989,0.189465,0.571429
1,3,1004,58.99,7,6,1,0.026892,0.058755,0.259259
2,1,162,14.00,4,5,4,0.006173,0.086420,4.000000
3,5,532,84.79,9,3,4,0.026316,0.159380,0.642857
4,5,890,42.16,4,0,0,0.021348,0.047371,0.210526
...,...,...,...,...,...,...,...,...,...
498116,4,783,59.10,2,2,0,0.012771,0.075479,0.200000
498117,1,278,98.90,9,5,4,0.050360,0.355755,0.642857
498118,3,300,5.41,6,6,4,0.030000,0.018033,0.666667
498119,2,1524,33.97,2,5,3,0.005906,0.022290,0.222222


In [585]:
y_test = pd.read_csv("dataset/DMC-2019-realclass.csv", sep="|")["fraud"]
y_test.value_counts()

fraud
0    474394
1     23727
Name: count, dtype: int64

In [586]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [587]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

In [588]:
y_train = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [589]:
BATCH_SIZE = 64

In [590]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=1024)

вес для редкого класса, чтобы модель его не пропускала

In [591]:
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

In [592]:
pos_weight = n_neg / n_pos
pos_weight

tensor(17.1084)

## Подготовка

In [593]:
load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "gp5")
WANDB_ENTITY = os.getenv("WANDB_ENTITY")


In [594]:
wandb.login(key=WANDB_API_KEY)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [595]:
def stop_logging():
    logger = logging.getLogger()
    for handler in logger.handlers:
        handler.flush()
        handler.close()
        logger.removeHandler(handler)

def new_log_file():
    stop_logging()
    timestamp = str(time.time()).replace('.', '_')
    log_file = f'part_2_{timestamp}.log'
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True
    )
    logging.info("Начал логгировать новый запуск")
    return log_file

In [629]:
def evaluate_model(model, X, y, threshold=0.5, name="dataset"):
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED) 

    model.eval()

    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)

    y_true = y.numpy().ravel()
    y_prob = probs.numpy().ravel()
    y_pred = (y_prob > threshold).astype(int)

    roc_auc = roc_auc_score(y_true, y_prob)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f05 = fbeta_score(y_true, y_pred, beta=0.5)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    profit = tp * 5 - fp * 25 - fn * 5

    print(name)
    print("ROC-AUC:", round(roc_auc, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F0.5-score:", round(f05, 4))
    print("Profit:", profit)
    print()

    return {
        "dataset": name,
        "threshold": threshold,
        "roc_auc": roc_auc,
        "precision": precision,
        "recall": recall,
        "f05": f05,
        "profit": profit
    }

In [630]:
def train_model(model, train_loader, X_valid, y_valid, loss_fn, optimizer, 
    epochs=30, threshold=0.5,model_name="model"):
    
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED) 

    best_metric = -10**9
    best_epoch = 0
    best_state = None

    history = []
    logging.info(f"Начали обучение {model_name}")

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0

        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)

        valid_metrics = evaluate_model(model, X_valid, y_valid, threshold=threshold,
            name=f"{model_name} | valid epoch {epoch}")
            
        current_metric = valid_metrics["roc_auc"]
        
        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        print(f"Эпоха {epoch} | " f"Train Loss: {epoch_loss:.4f} | " 
        f"Val ROC-AUC: {valid_metrics['roc_auc']:.4f} | " f"Val Profit: {valid_metrics['profit']}")
        
        
        logging.info(
            f"Эпоха {epoch} | "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Val ROC-AUC: {valid_metrics['roc_auc']:.4f} | "
            f"Val Profit: {valid_metrics['profit']}"
        )

        wandb.log({
             f"{model_name}/train_loss": epoch_loss,
             f"{model_name}/valid_profit": valid_metrics["profit"],
             f"{model_name}/valid_roc_auc": valid_metrics["roc_auc"],
             f"{model_name}/valid_precision": valid_metrics["precision"],
             f"{model_name}/valid_recall": valid_metrics["recall"],
             f"{model_name}/valid_f05": valid_metrics["f05"],
         })

    model.load_state_dict(best_state)

    print()
    print(f"Лучшая эпоха для {model_name}: {best_epoch}")
    print(f"Лучший ROC-AUC: {best_metric}")
    logging.info("Закончили обучение")
    logging.info(f"Лучшая эпоха для {model_name}: {best_epoch}")
    logging.info(f"Лучший ROC-AUC: {best_metric}")

    return model

In [627]:
def save_results(model, name, log_file, run):
    train_metrics = evaluate_model(model, X_train, y_train, threshold=0.5, name="Train")
    test_metrics = evaluate_model(model, X_test, y_test, threshold=0.5, name="Test")
    pickle.dump(model.state_dict(), open(f"models/{name}.pkl", 'wb'))
    logging.info("Сохранили веса модели в папку models")

    metric_keys = list(train_metrics.keys())

    table = wandb.Table(columns=metric_keys)
    table.add_data(*[train_metrics[i] for i in metric_keys])
    table.add_data(*[test_metrics[i] for i in metric_keys])
    wandb.log({'results': table})
    artifact = wandb.Artifact(name=name, type="model", description=f"Тест логгирования модели: {name}")

    artifact.add_file(f"models/{name}.pkl")
    artifact.add_file(log_file)
    run.log_artifact(artifact) 

## model_1_baseline

In [635]:
EPOCHS = 30
LR = 0.01
SEED = 42


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

model_1 = nn.Sequential(
    nn.Linear(9, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_1.parameters(), lr=LR)

config = {
    "model": "MLP",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_1)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_1_baseline", config=config)
log_file = new_log_file()

model_1 = train_model(
    model=model_1,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_1_baseline"
)

save_results(model_1, "model_1_baseline", log_file, run)


run.finish()

model_1_baseline | valid epoch 1
ROC-AUC: 0.9411
Precision: 0.25
Recall: 0.0952
F0.5-score: 0.1887
Profit: -235

Эпоха 1 | Train Loss: 1.2482 | Val ROC-AUC: 0.9411 | Val Profit: -235
model_1_baseline | valid epoch 2
ROC-AUC: 0.9553
Precision: 0.2188
Recall: 1.0
F0.5-score: 0.2593
Profit: -1770

Эпоха 2 | Train Loss: 0.7541 | Val ROC-AUC: 0.9553 | Val Profit: -1770
model_1_baseline | valid epoch 3
ROC-AUC: 0.972
Precision: 0.2561
Recall: 1.0
F0.5-score: 0.3009
Profit: -1420

Эпоха 3 | Train Loss: 0.4640 | Val ROC-AUC: 0.9720 | Val Profit: -1420
model_1_baseline | valid epoch 4
ROC-AUC: 0.9728
Precision: 0.2625
Recall: 1.0
F0.5-score: 0.3079
Profit: -1370

Эпоха 4 | Train Loss: 0.3839 | Val ROC-AUC: 0.9728 | Val Profit: -1370
model_1_baseline | valid epoch 5
ROC-AUC: 0.972
Precision: 0.2763
Recall: 1.0
F0.5-score: 0.3231
Profit: -1270

Эпоха 5 | Train Loss: 0.3468 | Val ROC-AUC: 0.9720 | Val Profit: -1270
model_1_baseline | valid epoch 6
ROC-AUC: 0.9741
Precision: 0.2838
Recall: 1.0
F0.5

model_1_baseline/train_loss,█▅▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_f05,▁▂▃▃▃▃▄▄▄▅▆▅▆▆▇▆▇▇▇▇▇█▇▆██▆▆█▆
model_1_baseline/valid_precision,▂▁▂▂▂▂▃▃▃▄▅▅▅▅▇▅▆▆▆▇▇█▆▆█▇▆▅█▆
model_1_baseline/valid_profit,█▁▃▃▃▄▄▅▅▅▆▆▆▆▇▆▇▇▇▇▇█▇▆▇▇▇▆█▆
model_1_baseline/valid_recall,▁██████████████████▇██████▇█▇█
model_1_baseline/valid_roc_auc,▁▃▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇█▇▇█▇███▆▆▇█
model_1_baseline/train_loss,0.1275
model_1_baseline/valid_f05,0.49296
model_1_baseline/valid_precision,0.4375
model_1_baseline/valid_profit,-570
model_1_baseline/valid_recall,1


In [636]:
train_metrics = evaluate_model(model_1, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_1, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9949
Precision: 0.4611
Recall: 1.0
F0.5-score: 0.5168
Profit: -2010

Test
ROC-AUC: 0.9861
Precision: 0.4143
Recall: 0.9676
F0.5-score: 0.4678
Profit: -700395



Неплохо, но видно сильное переобучение! Можно например, добавить регуляризацию, но так как это бейзлайн, то оставим так, а регуляризацию сделаем попозже

### Попробуем подобрать порог

In [ ]:
with torch.no_grad():
    val_prob = torch.sigmoid(model(X_val)).numpy()

val_true = y_val.numpy()

In [ ]:
from sklearn.metrics import precision_recall_curve

target_recall = 0.90

prec, rec, thr = precision_recall_curve(val_true, val_prob)
best_threshold = thr[rec[:-1] >= target_recall].max()
logging.info(f"Подобрали порог {best_threshold}")
wandb.log({"best_threshold": best_threshold})
best_threshold

In [ ]:
y_pred = (y_prob > best_threshold).astype(int)

In [ ]:
pre_at_score = precision_score(y_true, y_pred)
#logging.info(f"На тесте Precision: {pre_at_score}")
pre_at_score

In [ ]:
rec_at_score = recall_score(y_true, y_pred)
#logging.info(f"На тесте Recall: {rec_at_score}")
rec_at_score

In [ ]:
import matplotlib.pyplot as plt

plt.plot(rec, prec)
plt.xlabel("recall")
plt.ylabel("precision")
plt.title("PR кривая")
plt.show()

In [ ]:
#import pickle

#pickle.dump(model.state_dict(), open("models/model_fraud_1.pkl", 'wb'))
#logging.info("Сохранили веса модели в папку models")

In [ ]:
#results = wandb.Table(columns=['model', 'test/roc_auc', 'test/precision', 'test/recall', 'test/recall_new_threshold', 'test/precision_new_threshold'])
#results.add_data("nn_baseline", auc_score, pre_score, rec_score, rec_at_score, pre_at_score)
#wandb.log({"results": results})

In [ ]:
#artifact = wandb.Artifact(name="model_fraud_1", type="model", description="Тест логгирования модели MLP")

#artifact.add_file("models/model_fraud_1.pkl")
#artifact.add_file(log_file)
#run.log_artifact(artifact)




In [ ]:
#run.finish()

## model_2_dop_sloi

In [638]:
EPOCHS = 30
LR = 0.01

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_2 = nn.Sequential(
    nn.Linear(9, 32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.ReLU(),

    nn.Linear(16, 8),
    nn.ReLU(),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_2.parameters(), lr=LR)

config = {
    "model": "MLP_added_layer",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_2)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_2_added_layers", config=config)
log_file = new_log_file()

model_2 = train_model(
    model=model_2,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_2_dop_sloi"
)

save_results(model_2, "model_2", log_file, run)

run.finish()

model_2_dop_sloi/train_loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▂▁▁▃▂▁▁▁▁▁▂▁
model_2_dop_sloi/valid_f05,▂▂▁▂▂▂▃▃▄▃▃▃▄▄▄▆▆▆▆▄▅▂▄▄▇█▅█▆▆
model_2_dop_sloi/valid_precision,▂▂▁▂▂▂▃▃▄▃▃▃▄▄▄▆▅▆▆▃▅▂▄▄▆█▅█▆▆
model_2_dop_sloi/valid_profit,▄▃▁▃▃▃▄▅▆▅▅▅▆▆▆▇▇▇▇▅▇▃▆▆▇█▆█▇▇
model_2_dop_sloi/valid_recall,▄██▇█▇▇▇▁█▅█▇▄▅▅▅▅▄█▂███▅▄█▄▄▅
model_2_dop_sloi/valid_roc_auc,▁▄▃▅▅▅▅▅▅▅▅▆▇▆▆▇▇▇▇▆▆▅▆▇████▇█
model_2_dop_sloi/train_loss,0.15347
model_2_dop_sloi/valid_f05,0.6051
model_2_dop_sloi/valid_precision,0.55882
model_2_dop_sloi/valid_profit,-290
model_2_dop_sloi/valid_recall,0.90476


model_2_dop_sloi | valid epoch 1
ROC-AUC: 0.9311
Precision: 0.3051
Recall: 0.8571
F0.5-score: 0.3502
Profit: -950

Эпоха 1 | Train Loss: 1.1825 | Val ROC-AUC: 0.9311 | Val Profit: -950
model_2_dop_sloi | valid epoch 2
ROC-AUC: 0.9552
Precision: 0.3
Recall: 1.0
F0.5-score: 0.3488
Profit: -1120

Эпоха 2 | Train Loss: 0.6270 | Val ROC-AUC: 0.9552 | Val Profit: -1120
model_2_dop_sloi | valid epoch 3
ROC-AUC: 0.9502
Precision: 0.25
Recall: 1.0
F0.5-score: 0.2941
Profit: -1470

Эпоха 3 | Train Loss: 0.4225 | Val ROC-AUC: 0.9502 | Val Profit: -1470
model_2_dop_sloi | valid epoch 4
ROC-AUC: 0.9642
Precision: 0.2857
Recall: 0.9524
F0.5-score: 0.3322
Profit: -1155

Эпоха 4 | Train Loss: 0.4105 | Val ROC-AUC: 0.9642 | Val Profit: -1155
model_2_dop_sloi | valid epoch 5
ROC-AUC: 0.9641
Precision: 0.3182
Recall: 1.0
F0.5-score: 0.3684
Profit: -1020

Эпоха 5 | Train Loss: 0.3026 | Val ROC-AUC: 0.9641 | Val Profit: -1020
model_2_dop_sloi | valid epoch 6
ROC-AUC: 0.9599
Precision: 0.2941
Recall: 0.9524

model_2_dop_sloi/train_loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▂▁▁▃▂▁▁▁▁▁▂▁
model_2_dop_sloi/valid_f05,▂▂▁▂▂▂▃▃▄▃▃▃▄▄▄▆▆▆▆▄▅▂▄▄▇█▅█▆▆
model_2_dop_sloi/valid_precision,▂▂▁▂▂▂▃▃▄▃▃▃▄▄▄▆▅▆▆▃▅▂▄▄▆█▅█▆▆
model_2_dop_sloi/valid_profit,▄▃▁▃▃▃▄▅▆▅▅▅▆▆▆▇▇▇▇▅▇▃▆▆▇█▆█▇▇
model_2_dop_sloi/valid_recall,▄██▇█▇▇▇▁█▅█▇▄▅▅▅▅▄█▂███▅▄█▄▄▅
model_2_dop_sloi/valid_roc_auc,▁▄▃▅▅▅▅▅▅▅▅▆▇▆▆▇▇▇▇▆▆▅▆▇████▇█
model_2_dop_sloi/train_loss,0.15347
model_2_dop_sloi/valid_f05,0.6051
model_2_dop_sloi/valid_precision,0.55882
model_2_dop_sloi/valid_profit,-290
model_2_dop_sloi/valid_recall,0.90476


In [639]:
train_metrics = evaluate_model(model_2, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_2, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9974
Precision: 0.7345
Recall: 1.0
F0.5-score: 0.7757
Profit: -335

Test
ROC-AUC: 0.9854
Precision: 0.5884
Recall: 0.8528
F0.5-score: 0.6273
Profit: -270185



Да, добавив всего 1 доп слой, видно, как сильно улучшилась наша метрика Profit. С -700к поднялась до -270к

## model_3_bolshe_neyronov

In [644]:
EPOCHS = 30
LR = 0.01

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

model_3 = nn.Sequential(
    nn.Linear(9, 32),
    nn.ReLU(),
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_3.parameters(), lr=LR)

config = {
    "model": "MLP_added_neurons",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_3)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_3_more_neurons", config=config)
log_file = new_log_file()

model_3 = train_model(
    model=model_3,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_3_bolshe_neyronov"
)


save_results(model_3, "model_3", log_file, run)

run.finish()



model_3_bolshe_neyronov | valid epoch 1
ROC-AUC: 0.9586
Precision: 0.2442
Recall: 1.0
F0.5-score: 0.2877
Profit: -1520

Эпоха 1 | Train Loss: 0.9062 | Val ROC-AUC: 0.9586 | Val Profit: -1520
model_3_bolshe_neyronov | valid epoch 2
ROC-AUC: 0.9599
Precision: 0.3043
Recall: 1.0
F0.5-score: 0.3535
Profit: -1095

Эпоха 2 | Train Loss: 0.4444 | Val ROC-AUC: 0.9599 | Val Profit: -1095
model_3_bolshe_neyronov | valid epoch 3
ROC-AUC: 0.9612
Precision: 0.3279
Recall: 0.9524
F0.5-score: 0.3774
Profit: -930

Эпоха 3 | Train Loss: 0.3691 | Val ROC-AUC: 0.9612 | Val Profit: -930
model_3_bolshe_neyronov | valid epoch 4
ROC-AUC: 0.9679
Precision: 0.3922
Recall: 0.9524
F0.5-score: 0.4444
Profit: -680

Эпоха 4 | Train Loss: 0.3041 | Val ROC-AUC: 0.9679 | Val Profit: -680
model_3_bolshe_neyronov | valid epoch 5
ROC-AUC: 0.9658
Precision: 0.3704
Recall: 0.9524
F0.5-score: 0.4219
Profit: -755

Эпоха 5 | Train Loss: 0.2697 | Val ROC-AUC: 0.9658 | Val Profit: -755
model_3_bolshe_neyronov | valid epoch 6
RO

model_3_bolshe_neyronov/train_loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▂▃▂▁▁▁▁
model_3_bolshe_neyronov/valid_f05,▁▂▃▄▃▃▅▄▅▄▄▅▄▅▅▅▆▆▇▇▆▇█▄▄▇▇█▇█
model_3_bolshe_neyronov/valid_precision,▁▂▃▄▃▃▅▄▄▄▄▅▄▅▅▅▆▆▇▇▆▇█▄▄▆▇█▆█
model_3_bolshe_neyronov/valid_profit,▁▃▄▅▅▄▇▆▆▅▅▆▅▆▇▆▇▇██▇██▆▆▇██▇█
model_3_bolshe_neyronov/valid_recall,██▇▇▇▇▄▇▅█▇▅█▅▇▄▄▇▇▅▇▁▇▇▇▇▇▇▇▇
model_3_bolshe_neyronov/valid_roc_auc,▁▁▂▃▃▄▄▅▄▄▅▃▄▃▅▆▆▆▆▆▆▆▇▅▅▇▇█▇▇
model_3_bolshe_neyronov/train_loss,0.07161
model_3_bolshe_neyronov/valid_f05,0.67114
model_3_bolshe_neyronov/valid_precision,0.625
model_3_bolshe_neyronov/valid_profit,-205
model_3_bolshe_neyronov/valid_recall,0.95238


In [645]:
train_metrics = evaluate_model(model_3, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_3, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9981
Precision: 0.6803
Recall: 1.0
F0.5-score: 0.7268
Profit: -560

Test
ROC-AUC: 0.985
Precision: 0.5541
Recall: 0.8915
F0.5-score: 0.5995
Profit: -332630



Увеличение количества нейронов в 2 раза тоже улучшило качество отновительно бейзлайна, но не так сильно, как добавление доп слоя. 

Из интересного, если увеличить колво нейронов в 4 раза, то качество станет хуже 

## model_4_tolko_batchnorm

In [647]:
EPOCHS = 30
LR = 0.01

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_4 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),

    nn.Linear(8, 1)
)



run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_4_batchnorm_only", config=config)
log_file = new_log_file()

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_4.parameters(), lr=LR)

config = {
    "model": "MLP_batchnorm_only",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_4)
}

model_4 = train_model(
    model=model_4,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_4_tolko_batchnorm"
)


save_results(model_4, "model_4", log_file, run)

run.finish()

model_4_tolko_batchnorm/train_loss,█▅▄▃▃▂▂▃▂▂▂▂▂▁▃▃▂▂▂▁▂▂▂▂▁▁▁▃▂▂
model_4_tolko_batchnorm/valid_f05,▁▂▂▃▅▄▃▃▅▅▅▅█▆▂▄▅▆▇▇▅▅▅▆▇▇▅▃▃▄
model_4_tolko_batchnorm/valid_precision,▁▂▂▂▄▄▃▃▄▅▅▅█▇▂▃▅▅▇█▄▅▅▆▇█▆▃▃▄
model_4_tolko_batchnorm/valid_profit,▁▄▅▅▇▇▆▆▇▇▇▇██▅▆▇███▇▇▇████▆▆▇
model_4_tolko_batchnorm/valid_recall,█████▇███▆▇▅▅▂██▆▅▅▃▇▆▆▄▅▃▁███
model_4_tolko_batchnorm/valid_roc_auc,▆▇▆▇██▇█▇▇███▅▇██████▅▆▅▅▅▁█▇▇
model_4_tolko_batchnorm/train_loss,0.28975
model_4_tolko_batchnorm/valid_f05,0.45249
model_4_tolko_batchnorm/valid_precision,0.4
model_4_tolko_batchnorm/valid_profit,-655
model_4_tolko_batchnorm/valid_recall,0.95238


model_4_tolko_batchnorm | valid epoch 1
ROC-AUC: 0.9526
Precision: 0.1321
Recall: 1.0
F0.5-score: 0.1598
Profit: -3345

Эпоха 1 | Train Loss: 0.9802 | Val ROC-AUC: 0.9526 | Val Profit: -3345
model_4_tolko_batchnorm | valid epoch 2
ROC-AUC: 0.9611
Precision: 0.1963
Recall: 1.0
F0.5-score: 0.2339
Profit: -2045

Эпоха 2 | Train Loss: 0.5755 | Val ROC-AUC: 0.9611 | Val Profit: -2045
model_4_tolko_batchnorm | valid epoch 3
ROC-AUC: 0.9569
Precision: 0.2414
Recall: 1.0
F0.5-score: 0.2846
Profit: -1545

Эпоха 3 | Train Loss: 0.4234 | Val ROC-AUC: 0.9569 | Val Profit: -1545
model_4_tolko_batchnorm | valid epoch 4
ROC-AUC: 0.9654
Precision: 0.2727
Recall: 1.0
F0.5-score: 0.3191
Profit: -1295

Эпоха 4 | Train Loss: 0.4019 | Val ROC-AUC: 0.9654 | Val Profit: -1295
model_4_tolko_batchnorm | valid epoch 5
ROC-AUC: 0.9755
Precision: 0.4286
Recall: 1.0
F0.5-score: 0.4839
Profit: -595

Эпоха 5 | Train Loss: 0.3175 | Val ROC-AUC: 0.9755 | Val Profit: -595
model_4_tolko_batchnorm | valid epoch 6
ROC-AUC

model_4_tolko_batchnorm/train_loss,█▅▄▃▃▂▂▃▂▂▂▂▂▁▃▃▂▂▂▁▂▂▂▂▁▁▁▃▂▂
model_4_tolko_batchnorm/valid_f05,▁▂▂▃▅▄▃▃▅▅▅▅█▆▂▄▅▆▇▇▅▅▅▆▇▇▅▃▃▄
model_4_tolko_batchnorm/valid_precision,▁▂▂▂▄▄▃▃▄▅▅▅█▇▂▃▅▅▇█▄▅▅▆▇█▆▃▃▄
model_4_tolko_batchnorm/valid_profit,▁▄▅▅▇▇▆▆▇▇▇▇██▅▆▇███▇▇▇████▆▆▇
model_4_tolko_batchnorm/valid_recall,█████▇███▆▇▅▅▂██▆▅▅▃▇▆▆▄▅▃▁███
model_4_tolko_batchnorm/valid_roc_auc,▆▇▆▇██▇█▇▇███▅▇██████▅▆▅▅▅▁█▇▇
model_4_tolko_batchnorm/train_loss,0.28975
model_4_tolko_batchnorm/valid_f05,0.45249
model_4_tolko_batchnorm/valid_precision,0.4
model_4_tolko_batchnorm/valid_profit,-655
model_4_tolko_batchnorm/valid_recall,0.95238


In [648]:
train_metrics = evaluate_model(model_4, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_4, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.994
Precision: 0.8312
Recall: 0.7711
F0.5-score: 0.8184
Profit: -100

Test
ROC-AUC: 0.97
Precision: 0.7059
Recall: 0.6019
F0.5-score: 0.6823
Profit: -124565



Так как в предыдущие разы качество улучшилось с увеличением колво слоев, то теперь мы решили добавить колво слоев, а также после каждого слоя сделать только BatchNorm

Видно, что это сильно улучшило качество!

## model_5_tolko_dropout

In [653]:
EPOCHS = 40
LR = 0.01
DROPOUT_COEF = 0.1

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_5 = nn.Sequential(
    nn.Linear(9, 128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_5.parameters(), lr=LR)

config = {
    "model": "MLP_dropout_only",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_5)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_5_dropout_only", config=config)
log_file = new_log_file()

model_5 = train_model(
    model=model_5,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_5_tolko_dropout"
)

save_results(model_5, "model_5", log_file, run)

run.finish()


model_5_tolko_dropout | valid epoch 1
ROC-AUC: 0.8844
Precision: 0.1944
Recall: 1.0
F0.5-score: 0.2318
Profit: -2070

Эпоха 1 | Train Loss: 1.1128 | Val ROC-AUC: 0.8844 | Val Profit: -2070
model_5_tolko_dropout | valid epoch 2
ROC-AUC: 0.9249
Precision: 0.1909
Recall: 1.0
F0.5-score: 0.2278
Profit: -2120

Эпоха 2 | Train Loss: 0.8482 | Val ROC-AUC: 0.9249 | Val Profit: -2120
model_5_tolko_dropout | valid epoch 3
ROC-AUC: 0.9371
Precision: 0.3409
Recall: 0.7143
F0.5-score: 0.3807
Profit: -680

Эпоха 3 | Train Loss: 0.5357 | Val ROC-AUC: 0.9371 | Val Profit: -680
model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.9647
Precision: 0.3043
Recall: 1.0
F0.5-score: 0.3535
Profit: -1095

Эпоха 4 | Train Loss: 0.4602 | Val ROC-AUC: 0.9647 | Val Profit: -1095
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9548
Precision: 0.3175
Recall: 0.9524
F0.5-score: 0.3663
Profit: -980

Эпоха 5 | Train Loss: 0.3578 | Val ROC-AUC: 0.9548 | Val Profit: -980
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.9603

model_5_tolko_dropout/train_loss,█▆▄▄▃▃▃▃▃▃▃▃▃▂▂▂▃▃▃▂▂▂▂▃▃▂▂▁▂▂▂▂▂▃▂▁▁▂▁▁
model_5_tolko_dropout/valid_f05,▁▁▄▃▃▃▅▃▂▅▂▂▂▄▄▃▃▃▃▃▅▅▆▃▃▅▇▆▄▅▄▇▇▃▅▇▅▆██
model_5_tolko_dropout/valid_precision,▁▁▃▃▃▃▄▃▂▄▂▂▂▄▄▃▃▃▃▃▄▄▆▃▃▅▇▆▄▄▄▇▇▂▄▇▅▆██
model_5_tolko_dropout/valid_profit,▁▁▆▅▅▅▆▅▄▆▃▃▄▆▆▅▅▅▅▅▆▆▇▄▅▇▇▇▆▆▆▇▇▅▆▇▇▇██
model_5_tolko_dropout/valid_recall,██▃█▇███▇▇███▅▇▆▆▇███▇▇██▆▅▆▆▆▇▆▅▅▆▅▆▆▄▁
model_5_tolko_dropout/valid_roc_auc,▁▄▅▇▆▇▇▇▆▇▇▇▆▇▇▇▆▇█▇█▇▇▇▇███▇████▅▇█▇▇██
model_5_tolko_dropout/train_loss,0.0531
model_5_tolko_dropout/valid_f05,0.61905
model_5_tolko_dropout/valid_precision,0.61905
model_5_tolko_dropout/valid_profit,-175
model_5_tolko_dropout/valid_recall,0.61905


In [654]:
train_metrics = evaluate_model(model_5, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_5, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9895
Precision: 0.7019
Recall: 0.8795
F0.5-score: 0.7315
Profit: -460

Test
ROC-AUC: 0.9788
Precision: 0.5378
Recall: 0.7792
F0.5-score: 0.5733
Profit: -330965



Теперь сделаем эксперимет с dropout. Увеличим колво эпох и попробуем подобрать коэффицент экспериментально. Лучшее качество достигли при коэффиценте = 0.1

Увеличение колво эпох не улучшило ситуацию с переобучением

Это лучше чем бейзлайн, но батнорм показал себя лучше в 3 раза чем дропаут в этом эксперименте

## model_6_batchnorm_i_dropout

In [661]:
EPOCHS = 30
LR = 0.01
DROPOUT_COEF = 0.1
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_6 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_6.parameters(), lr=LR)

config = {
    "model": "MLP_batchnorm_dropout",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_6)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_6_batchnorm_dropout", config=config)
log_file = new_log_file()

model_6 = train_model(
    model=model_6,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_6_batchnorm_i_dropout"
)


save_results(model_6, "model_6", log_file, run)

run.finish()

model_6_batchnorm_i_dropout | valid epoch 1
ROC-AUC: 0.9462
Precision: 0.1207
Recall: 1.0
F0.5-score: 0.1464
Profit: -3720

Эпоха 1 | Train Loss: 1.1907 | Val ROC-AUC: 0.9462 | Val Profit: -3720
model_6_batchnorm_i_dropout | valid epoch 2
ROC-AUC: 0.9659
Precision: 0.1641
Recall: 1.0
F0.5-score: 0.197
Profit: -2570

Эпоха 2 | Train Loss: 0.7817 | Val ROC-AUC: 0.9659 | Val Profit: -2570
model_6_batchnorm_i_dropout | valid epoch 3
ROC-AUC: 0.9602
Precision: 0.2941
Recall: 0.9524
F0.5-score: 0.3413
Profit: -1105

Эпоха 3 | Train Loss: 0.4925 | Val ROC-AUC: 0.9602 | Val Profit: -1105
model_6_batchnorm_i_dropout | valid epoch 4
ROC-AUC: 0.9486
Precision: 0.2234
Recall: 1.0
F0.5-score: 0.2645
Profit: -1720

Эпоха 4 | Train Loss: 0.4173 | Val ROC-AUC: 0.9486 | Val Profit: -1720
model_6_batchnorm_i_dropout | valid epoch 5
ROC-AUC: 0.9663
Precision: 0.2958
Recall: 1.0
F0.5-score: 0.3443
Profit: -1145

Эпоха 5 | Train Loss: 0.3899 | Val ROC-AUC: 0.9663 | Val Profit: -1145
model_6_batchnorm_i_dro

model_6_batchnorm_i_dropout/train_loss,█▆▄▃▃▃▂▂▂▂▂▃▂▂▂▂▁▁▁▁▂▂▂▂▁▁▁▁▁▁
model_6_batchnorm_i_dropout/valid_f05,▁▂▄▃▄▅▇▄▇█▃▄▇██▅▆▇█▄▅▆█▆▆▄▅▄▄▄
model_6_batchnorm_i_dropout/valid_precision,▁▂▃▂▃▄▆▄▇▇▃▃▅▇▇▅▆██▅▄▅▇▆▆▅▇▆▆▆
model_6_batchnorm_i_dropout/valid_profit,▁▃▆▅▆▇█▆██▅▆▇███████▇▇████████
model_6_batchnorm_i_dropout/valid_recall,██████▇█▄▄███▆▆▂▂▃▃▁▇▆▅▃▂▁▁▁▁▁
model_6_batchnorm_i_dropout/valid_roc_auc,▆▇▇▆▇███▄▁▇▇███▇▇▆▇▂▆▆▇██▇████
model_6_batchnorm_i_dropout/train_loss,0.02528
model_6_batchnorm_i_dropout/valid_f05,0.37736
model_6_batchnorm_i_dropout/valid_precision,0.5
model_6_batchnorm_i_dropout/valid_profit,-165
model_6_batchnorm_i_dropout/valid_recall,0.19048


In [662]:
train_metrics = evaluate_model(model_6, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_6, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9804
Precision: 0.9565
Recall: 0.2651
F0.5-score: 0.6286
Profit: -220

Test
ROC-AUC: 0.9489
Precision: 0.8013
Recall: 0.2525
F0.5-score: 0.5585
Profit: -95860



Если соединить 2 и дропаут и батчнорм вместе видим хорошее качество -- наш Profit поднялся до -95к

## model_7_leaky_relu

In [677]:
EPOCHS = 50
LR = 0.01
DROPOUT_COEF = 0.3
NEG_SLOPE = 0.03

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_7 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_7.parameters(), lr=LR)

config = {
    "model": "MLP_leaky_relu",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "negative_slope": NEG_SLOPE,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_7)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_7_leaky_relu", config=config)
log_file = new_log_file()

model_7 = train_model(
    model=model_7,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_7_leaky_relu"
)
save_results(model_7, "model_7", log_file, run)

run.finish()

model_7_leaky_relu | valid epoch 1
ROC-AUC: 0.9484
Precision: 0.1391
Recall: 1.0
F0.5-score: 0.168
Profit: -3145

Эпоха 1 | Train Loss: 1.1772 | Val ROC-AUC: 0.9484 | Val Profit: -3145
model_7_leaky_relu | valid epoch 2
ROC-AUC: 0.9353
Precision: 0.1654
Recall: 1.0
F0.5-score: 0.1985
Profit: -2545

Эпоха 2 | Train Loss: 0.8285 | Val ROC-AUC: 0.9353 | Val Profit: -2545
model_7_leaky_relu | valid epoch 3
ROC-AUC: 0.9581
Precision: 0.4286
Recall: 0.7143
F0.5-score: 0.4658
Profit: -455

Эпоха 3 | Train Loss: 0.5045 | Val ROC-AUC: 0.9581 | Val Profit: -455
model_7_leaky_relu | valid epoch 4
ROC-AUC: 0.9319
Precision: 0.3208
Recall: 0.8095
F0.5-score: 0.3648
Profit: -835

Эпоха 4 | Train Loss: 0.3032 | Val ROC-AUC: 0.9319 | Val Profit: -835
model_7_leaky_relu | valid epoch 5
ROC-AUC: 0.9496
Precision: 0.3077
Recall: 0.9524
F0.5-score: 0.3559
Profit: -1030

Эпоха 5 | Train Loss: 0.3013 | Val ROC-AUC: 0.9496 | Val Profit: -1030
model_7_leaky_relu | valid epoch 6
ROC-AUC: 0.928
Precision: 0.365

model_7_leaky_relu/train_loss,█▆▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_7_leaky_relu/valid_f05,▃▄▇▆▆▆██▇█▇█▇▇▃▃▁▁▃▃▁▁▁▁▁▁▁▃▃▁▁▁▁▁▁▁▁▁▁▁
model_7_leaky_relu/valid_precision,▃▃▆▅▅▅▆▇▆▇███▄▄▁▄▄▁▁▁▁▁▁▁▁▁▅▅▁▁▁▁▁▁▁▁▁▁▁
model_7_leaky_relu/valid_profit,▁▂▇▆▆▇▇▆▇███████████████████████████████
model_7_leaky_relu/valid_recall,█▆▇█▆▇▇█▇▅▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_7_leaky_relu/valid_roc_auc,▅▃▇▃▅▁▇▃▇████▇▇▇▆▆▆▆▇▇▇▇▆▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇
model_7_leaky_relu/train_loss,0.00338
model_7_leaky_relu/valid_f05,0
model_7_leaky_relu/valid_precision,0
model_7_leaky_relu/valid_profit,-155
model_7_leaky_relu/valid_recall,0


In [678]:
train_metrics = evaluate_model(model_7, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_7, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.977
Precision: 0.6515
Recall: 0.5181
F0.5-score: 0.6196
Profit: -560

Test
ROC-AUC: 0.9631
Precision: 0.5724
Recall: 0.4502
F0.5-score: 0.5429
Profit: -211275



Поменяв функцию активации с ReLu на LeakyRelu и поиграв с параметрами EPOCHS, DROPOUT_COEF и
NEG_SLOPE получилось добиться лучшего качества по Profit в -211к

Да, это лучше бейзлайна, но качество ухудшилось. 

Вывод: особого эффекта LeakyRelu не дал

## model_8_weight_decay

In [716]:
EPOCHS = 40
LR = 0.01
WEIGHT_DECAY=0.001
DROPOUT_COEF = 0.3

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_8 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_8.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

config = {
    "model": "MLP_weight_decay",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_8)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_8_weight_decay", config=config)
log_file = new_log_file()

model_8 = train_model(
    model=model_8,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_8_weight_decay"
)

save_results(model_8, "model_8", log_file, run)

run.finish()

model_8_weight_decay | valid epoch 1
ROC-AUC: 0.9199
Precision: 0.1105
Recall: 1.0
F0.5-score: 0.1344
Profit: -4120

Эпоха 1 | Train Loss: 1.2208 | Val ROC-AUC: 0.9199 | Val Profit: -4120
model_8_weight_decay | valid epoch 2
ROC-AUC: 0.9528
Precision: 0.1567
Recall: 1.0
F0.5-score: 0.1885
Profit: -2720

Эпоха 2 | Train Loss: 0.9239 | Val ROC-AUC: 0.9528 | Val Profit: -2720
model_8_weight_decay | valid epoch 3
ROC-AUC: 0.963
Precision: 0.3704
Recall: 0.9524
F0.5-score: 0.4219
Profit: -755

Эпоха 3 | Train Loss: 0.6171 | Val ROC-AUC: 0.9630 | Val Profit: -755
model_8_weight_decay | valid epoch 4
ROC-AUC: 0.9427
Precision: 0.2778
Recall: 0.7143
F0.5-score: 0.3165
Profit: -930

Эпоха 4 | Train Loss: 0.4336 | Val ROC-AUC: 0.9427 | Val Profit: -930
model_8_weight_decay | valid epoch 5
ROC-AUC: 0.9199
Precision: 0.25
Recall: 0.5714
F0.5-score: 0.2817
Profit: -885

Эпоха 5 | Train Loss: 0.4106 | Val ROC-AUC: 0.9199 | Val Profit: -885
model_8_weight_decay | valid epoch 6
ROC-AUC: 0.959
Precisio

model_8_weight_decay/train_loss,█▆▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▇▄▃▂▂▂▁▁▁▁▁▁▁▁
model_8_weight_decay/valid_f05,▃▃▆▅▄▅█▆▇▇▆█▇█▆▄▄▃▃▃▃▃▁▁▇▅▄▅▆█▆▆▅▆▆▆▅▅▅▄
model_8_weight_decay/valid_precision,▂▂▄▄▃▄▆▄▅▅▅▇▆██▇▇▅▅▅▅▅▁▁▇▄▃▄▄▆▅▅▅▅▇▆▅▅▅▅
model_8_weight_decay/valid_profit,▁▃▇▇▇▆█▇▇▇███████████████▆▆▆▇███████████
model_8_weight_decay/valid_recall,███▆▅█▆▅▇▇▃▄▃▃▂▂▂▁▁▁▁▁▁▁▃▇▆█▇▆▅▄▃▃▃▂▂▂▂▂
model_8_weight_decay/valid_roc_auc,▂▆▇▄▂▆▅▅██▄▁▇█▇▅▇▇▇▇▇▇▇█▇▂▁▇▇▇▇▇▇▇▇▇▇▇▇▇
model_8_weight_decay/train_loss,0.02826
model_8_weight_decay/valid_f05,0.27027
model_8_weight_decay/valid_precision,0.5
model_8_weight_decay/valid_profit,-135
model_8_weight_decay/valid_recall,0.09524


In [717]:
train_metrics = evaluate_model(model_8, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_8, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9611
Precision: 0.85
Recall: 0.2048
F0.5-score: 0.5215
Profit: -320

Test
ROC-AUC: 0.9449
Precision: 0.7599
Recall: 0.2423
F0.5-score: 0.5324
Profit: -106555



Наконец-то добавив регуляризацию видим, что качество значительно улучшилось относительно бейзлайна, но ухудшилось относительно эксперимента с batchnorm и dropout одновременно

При этом пришлось обратно вернуться к обычной RELU

## model_9_Focal_loss

In [698]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        targets = targets.float()

        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        loss = focal_weight * bce_loss

        return loss.mean()

In [746]:
EPOCHS = 30
LR = 0.01
DROPOUT_COEF = 0.1
SEED = 42
WEIGHT_DECAY = 0.0001
ALPHA = 0.99
GAMMA = 2.0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_9 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = FocalLoss(alpha=ALPHA, gamma=GAMMA)
optimizer = torch.optim.Adam(model_9.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

config = {
    "model": "MLP_focal_loss",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "alpha": ALPHA,
    "gamma": GAMMA,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_8)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_9_focal_loss", config=config)
log_file = new_log_file()

model_9 = train_model(
    model=model_9,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_9_Focal_loss"
)


save_results(model_9, "model_9", log_file, run)

run.finish()

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

model_9_Focal_loss | valid epoch 1
ROC-AUC: 0.9325
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 1 | Train Loss: 0.0800 | Val ROC-AUC: 0.9325 | Val Profit: -105
model_9_Focal_loss | valid epoch 2
ROC-AUC: 0.9219
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 2 | Train Loss: 0.0371 | Val ROC-AUC: 0.9219 | Val Profit: -105
model_9_Focal_loss | valid epoch 3
ROC-AUC: 0.9512
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 3 | Train Loss: 0.0281 | Val ROC-AUC: 0.9512 | Val Profit: -105
model_9_Focal_loss | valid epoch 4
ROC-AUC: 0.9548
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 4 | Train Loss: 0.0292 | Val ROC-AUC: 0.9548 | Val Profit: -105
model_9_Focal_loss | valid epoch 5
ROC-AUC: 0.9552
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 5 | Train Loss: 0.0258 | Val ROC-AUC: 0.9552 | Val Profit: -105
model_9_Focal_loss | valid epoch 6
ROC-AUC: 0.9495
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эп

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

model_9_Focal_loss | valid epoch 7
ROC-AUC: 0.9537
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 7 | Train Loss: 0.0266 | Val ROC-AUC: 0.9537 | Val Profit: -105
model_9_Focal_loss | valid epoch 8
ROC-AUC: 0.9571
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 8 | Train Loss: 0.0216 | Val ROC-AUC: 0.9571 | Val Profit: -105
model_9_Focal_loss | valid epoch 9
ROC-AUC: 0.9658
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 9 | Train Loss: 0.0206 | Val ROC-AUC: 0.9658 | Val Profit: -105
model_9_Focal_loss | valid epoch 10
ROC-AUC: 0.9661
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 10 | Train Loss: 0.0176 | Val ROC-AUC: 0.9661 | Val Profit: -105
model_9_Focal_loss | valid epoch 11
ROC-AUC: 0.9353
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 11 | Train Loss: 0.0200 | Val ROC-AUC: 0.9353 | Val Profit: -105
model_9_Focal_loss | valid epoch 12
ROC-AUC: 0.9521
Precision: 0.5385
Recall: 0.3333
F0.5-score: 0.4795
Pr

model_9_Focal_loss/train_loss,█▄▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂▁▁▂▂▁▁▁
model_9_Focal_loss/valid_f05,▁▁▁▁▁▁▁▁▁▁▁▆▆▆▆▇▇▇▇▅▇▇▆▇▆▇█▃▄▇
model_9_Focal_loss/valid_precision,▁▁▁▁▁▁▁▁▁▁▁▇▆▇█▇██▆▆▇█▇▇█▆█▅██
model_9_Focal_loss/valid_profit,███████████▆▂▇▇▄▇▇▁▅▇▇▆▆▇▂▆▇█▇
model_9_Focal_loss/valid_recall,▁▁▁▁▁▁▁▁▁▁▁▄▆▄▃▆▅▄█▃▄▄▄▅▃▇▇▁▂▄
model_9_Focal_loss/valid_roc_auc,▅▄▆▆▆▆▆▆▇▇▅▆▁█▆▇██▇▇▇▄▇▅▅▇██▇█
model_9_Focal_loss/train_loss,0.00748
model_9_Focal_loss/valid_f05,0.53846
model_9_Focal_loss/valid_precision,0.63636
model_9_Focal_loss/valid_profit,-135
model_9_Focal_loss/valid_recall,0.33333


In [747]:
train_metrics = evaluate_model(model_9, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_9, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9734
Precision: 0.8125
Recall: 0.1566
F0.5-score: 0.4422
Profit: -360

Test
ROC-AUC: 0.969
Precision: 0.8491
Recall: 0.1888
F0.5-score: 0.4997
Profit: -93735



В этом эксперименте пробовали ментять функцию активации, но это особо не помогло. Использовал FocalLoss

Параметр альфа = вес класса 1. Поставил его побольше, так как у нас 1 в 17 раз меньше чем 0. Нам важны ошибки, которые модель допускает на 1. 

Параметр гамма = Насколько быстро модель забывает "легкие" нули и фокусируется на сложных границах между классами. Обычно ставят по умолчанию 2.

В случае FocalLoss мы мало штрафуем за уверенные ответы и сильно штрафуем за неуверенные. Отлично подходит для задач, когда сильный дизбаланс классов. 


Но в нашем случае это не дало сильного прироста эффекта. Получилось улучшить на 2к всего от нашего лучшего предыдущего эффекта. Но тем не менее эта архитектура наиболее эффективной получилась сейчас 

# ансамбль ПОКА НЕ ТРОГАЛ

In [ ]:
import numpy as np

In [ ]:
def build_model():
    return nn.Sequential(
        nn.Linear(9, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(32, 1),
        )

In [ ]:
EPOCHS = 120
LR = 0.001
WEIGHT_DECAY = 1e-4
N_ENSEMBLE = 5

net = build_model()
opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
loss_fn2 = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
config = {
    "model": "Ensemble",
    "optimizer": str(opt.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pos_weight": pos_weight,
    "loss": str(loss_fn2.__class__.__name__),
    "n_ensemble": N_ENSEMBLE,
    "architecture": str(net)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="ensemble", config=config)

In [ ]:
test_probs = []
logging.info("Запустили обучение моделей ансамбля")

np.random.seed(42)
seeds = np.random.randint(1, 525252, size=N_ENSEMBLE)
ensemble = []
for i in range(N_ENSEMBLE):
    seed = seeds[i]
    torch.manual_seed(seed)
    net = build_model()
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    logging.info(f"Запустили обучение модели {i+1}. Сид: {seed}")


    net.train()
    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn2(net(xb), yb)
            loss.backward()
            epoch_loss += loss.item()
            opt.step()
        
        logging.info(f"эпоха: {epoch}, loss:  {round(epoch_loss, 4)}")
        wandb.log({"epoch": epoch, f"ensemble/{i}/train_loss": epoch_loss})
    
    ensemble.append(net.state_dict())

    net.eval()
    with torch.no_grad():
        test_probs.append(torch.sigmoid(net(X_test)).numpy())

    logging.info(f"Сеть {i} обучена")
    print("сеть", i, "обучена")

probs2 = np.mean(test_probs, axis=0)

In [ ]:
auc_score = roc_auc_score(y_true, probs2)
logging.info(f"На тесте ROC_AUC: {auc_score}")
auc_score

In [ ]:
pre_score = average_precision_score(y_true, probs2)
logging.info(f"На тесте Precision: {pre_score}")
pre_score

In [ ]:
results = wandb.Table(columns=['model', 'test/avg_roc_auc', 'test/avg_precision'])
results.add_data("ensemble", auc_score, pre_score)
wandb.log({"results": results})

In [ ]:
import pickle

pickle.dump(ensemble, open("models/model_fraud_ensemble.pkl", 'wb'))
logging.info("Сохранили веса моделей ансамбля в папку models")

In [ ]:
artifact = wandb.Artifact(name="model_fraud_ensemble.pkl", type="model", description="Ансамбль моделей для определения Фрода")

artifact.add_file("models/model_fraud_ensemble.pkl")
artifact.add_file(log_file)
run.log_artifact(artifact)


In [ ]:
run.finish()

In [36]:
wandb.finish()


In [608]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np

k_values = [3, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]

best_k = None
best_roc_auc = -1
best_knn = None

X_train_np = X_train.numpy()
y_train_np = y_train.numpy().ravel()

X_val_np = X_val.numpy()
y_val_np = y_val.numpy().ravel()

X_test_np = X_test.numpy()
y_test_np = y_test.numpy().ravel()

for k in k_values:
    knn = KNeighborsClassifier(
        n_neighbors=k,
        weights="distance"
    )

    knn.fit(X_train_np, y_train_np)

    val_probs = knn.predict_proba(X_val_np)[:, 1]
    val_roc_auc = roc_auc_score(y_val_np, val_probs)

    print(f"k={k} | val ROC-AUC={val_roc_auc:.4f}")

    if val_roc_auc > best_roc_auc:
        best_roc_auc = val_roc_auc
        best_k = k
        best_knn = knn

print("\nЛучший KNN")
print("-" * 30)
print("best k:", best_k)
print("best val ROC-AUC:", round(best_roc_auc, 4))

k=3 | val ROC-AUC=0.7783
k=5 | val ROC-AUC=0.8613
k=7 | val ROC-AUC=0.8891
k=8 | val ROC-AUC=0.9312
k=9 | val ROC-AUC=0.9356
k=10 | val ROC-AUC=0.9371
k=11 | val ROC-AUC=0.9336
k=12 | val ROC-AUC=0.9298
k=13 | val ROC-AUC=0.9272
k=14 | val ROC-AUC=0.9328
k=15 | val ROC-AUC=0.9313

Лучший KNN
------------------------------
best k: 10
best val ROC-AUC: 0.9371


In [609]:
threshold = 0.5

test_probs = best_knn.predict_proba(X_test_np)[:, 1]
test_pred = (test_probs >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_np, test_pred).ravel()

test_profit = tp * 5 - fp * 25 - fn * 5
test_roc_auc = roc_auc_score(y_test_np, test_probs)

print("\nKNN на test")
print("-" * 30)
print("k:", best_k)
print("threshold:", threshold)
print("ROC-AUC:", round(test_roc_auc, 4))
print("Profit:", test_profit)
print("TN, FP, FN, TP:", tn, fp, fn, tp)


KNN на test
------------------------------
k: 10
threshold: 0.5
ROC-AUC: 0.8991
Profit: -130230
TN, FP, FN, TP: 473649 745 23024 703
